# Diyabet tahmin

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

In [2]:
train= pd.read_csv('train(4).csv')
test= pd.read_csv('test(4).csv')

In [3]:
train.head()

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
0,0,31,1,45,7.7,6.8,6.1,33.4,0.93,112,...,Female,Hispanic,Highschool,Lower-Middle,Current,Employed,0,0,0,1.0
1,1,50,2,73,5.7,6.5,5.8,23.8,0.83,120,...,Female,White,Highschool,Upper-Middle,Never,Employed,0,0,0,1.0
2,2,32,3,158,8.5,7.4,9.1,24.1,0.83,95,...,Male,Hispanic,Highschool,Lower-Middle,Never,Retired,0,0,0,0.0
3,3,54,3,77,4.6,7.0,9.2,26.6,0.83,121,...,Female,White,Highschool,Lower-Middle,Current,Employed,0,1,0,1.0
4,4,54,1,55,5.7,6.2,5.1,28.8,0.90,108,...,Male,White,Highschool,Upper-Middle,Never,Retired,0,1,0,1.0


In [4]:
test.head()

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,triglycerides,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history
0,700000,45,4,100,4.3,6.8,6.2,25.5,0.84,123,...,111,Female,White,Highschool,Middle,Former,Employed,0,0,0
1,700001,35,1,87,3.5,4.6,9.0,28.6,0.88,120,...,145,Female,White,Highschool,Middle,Never,Unemployed,0,0,0
2,700002,45,1,61,7.6,6.8,7.0,28.5,0.94,112,...,184,Male,White,Highschool,Low,Never,Employed,0,0,0
3,700003,55,2,81,7.3,7.3,5.0,26.9,0.91,114,...,128,Male,White,Graduate,Middle,Former,Employed,0,0,0
4,700004,77,2,29,7.3,7.6,8.5,22.0,0.83,131,...,133,Male,White,Graduate,Low,Current,Unemployed,0,0,0


In [5]:
HEDEF = 'diagnosed_diabetes'

In [6]:
# 2. Kategorik (Metinsel) Sütunları Güvenle Sayısallaştırma
# Sadece girdi özelliklerindeki metin sütunlarını seçiyoruz, hedef kolonu dışarıda bırakıyoruz
#Çok sayıdaki kategorik (metinsel) sütunu (gender, ethnicity, education_level, smoking_status, family_history_diabetes vb.)
#OrdinalEncoder ile güvenli şekilde sayısallaştırmak
kategorik_sutunlar = [
    kolon for kolon in train.select_dtypes(include=['object']).columns 
    if kolon != HEDEF
]

In [7]:
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train[kategorik_sutunlar] = encoder.fit_transform(train[kategorik_sutunlar].astype(str))
test[kategorik_sutunlar] = encoder.transform(test[kategorik_sutunlar].astype(str))


In [8]:
# 3. Sağlık Bilimi feature engineer (Diyabet ve Metabolizma İndeksleri)
#Metabolik Risk İndeksi: Bel-kalça oranı ile BMI değerinin çarpımı (waist_to_hip_ratio * bmi).
#Hareketsizlik Dengesi: Haftalık ekran süresinin fiziksel aktivite süresine oranı (screen_time_hours_per_day / physical_activity_minutes_per_week)
#Kardiyovasküler Yük: Sistolik kan basıncı ile trigliserit seviyesinin etkileşimi (systolic_bp * triglycerides)

for df in [train, test]:
    # Bel-Kalça Oranı ve BMI Kombinasyonu (Metabolik Sendrom Belirteci)
    df['metabolic_risk_index'] = df['waist_to_hip_ratio'] * df['bmi']
    
    # Hareketsizlik Endeksi (Ekran süresi artıp fiziksel aktivite düştükçe risk artar)
    df['sedentary_lifestyle_index'] = (df['screen_time_hours_per_day'] * 7) / (df['physical_activity_minutes_per_week'] + 1e-5)
    
    # Klinik Risk (Tansiyon ve Trigliserit çarpımı)
    # Eğer trigliserit sütunu train veya test içinde değişkenlik gösteriyorsa koruma amaçlı 1e-5 ekliyoruz
    if 'triglycerides' in df.columns:
        df['bp_lipid_interaction'] = df['systolic_bp'] * df['triglycerides']
    else:
        df['bp_lipid_interaction'] = df['systolic_bp'] * 1.0

print("2. Adım: Metabolik özellik mühendisliği tamamlandı. Modelleme başlatılıyor...")


2. Adım: Metabolik özellik mühendisliği tamamlandı. Modelleme başlatılıyor...


In [9]:
# 4. Girdileri (X) ve Hedefi (y) Kesin Olarak Ayırma
giris_sutunlari = [kolon for kolon in train.columns if kolon not in ['id', HEDEF]]

X = train[giris_sutunlari]
y = train[HEDEF]
X_test = test[giris_sutunlari] # Train ile tamamen aynı sütunlar seçildi, hata vermez!


In [10]:
# 5. 5-Fold Stratified Cross Validation Düzeni
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_tahminleri = np.zeros(len(train))
test_tahminleri = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # Sınıflandırma için LightGBM kurulumu
    model = LGBMClassifier(
        n_estimators=500,
        learning_rate=0.04,
        random_state=42,
        verbose=-1
    )
    model.fit(X_train, y_train)
    
    # ROC-AUC için 1 sınıfına ait olasılıkları tahmin ediyoruz (predict_proba[:, 1])
    oof_tahminleri[val_idx] = model.predict_proba(X_val)[:, 1]
    test_tahminleri += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Katman skoru hesaplama
    fold_auc = roc_auc_score(y_val, oof_tahminleri[val_idx])
    print(f"Katman {fold + 1} ROC-AUC Skoru: {fold_auc:.4f}")

# Genel Başarı Oranı
genel_auc = roc_auc_score(y, oof_tahminleri)
print(f"\n---> TÜM VERİ SETİ GENEL ROC-AUC SKORU: {genel_auc:.4f} <---")


Katman 1 ROC-AUC Skoru: 0.7254
Katman 2 ROC-AUC Skoru: 0.7233
Katman 3 ROC-AUC Skoru: 0.7239
Katman 4 ROC-AUC Skoru: 0.7253
Katman 5 ROC-AUC Skoru: 0.7248

---> TÜM VERİ SETİ GENEL ROC-AUC SKORU: 0.7245 <---


In [11]:
# 6. Kaggle Gönderi Dosyasının Oluşturulması
submission = pd.DataFrame({
    'id': test['id'],
    'diagnosed_diabetes': test_tahminleri
})

In [12]:
submission.to_csv('submission_diabetes.csv', index=False)